In [ ]:
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "src").exists():
        repo_root = candidate.resolve()
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        break
if repo_root is None:
    raise RuntimeError("Could not locate project root containing 'src'.")

In [ ]:
# Parameters
model_name = None  # Optional override; normally inferred from selected config
TICKERS = ["AAPL", "INTC", "LCID"]
SEEDS = [4718, 4819]
num_epochs = 5
LEARNING_RATE = 5e-4
PREPROCESSING_CONTRACT_VERSION = 1
MAX_SAMPLES_PER_ORDER = 100
SPLIT_CAP_RANDOM_SEED = 4718
NUM_TIME_STEPS = 30

WANDB_ENABLE = True
WANDB_PROJECT = "arch-grid-search-static"
WANDB_MODE = "online"

ORDERS_PER_BATCH = 64
BATCH_SIZE = 1024
SIGMA = 0.1
ALPHA = 1.0
BETA_L3 = 0.2

DEFAULT_CONFIG_NAME = "deephit_configs_param_200k_500k_v1.jsonl"
CONFIG_JSONL_PATH = Path(f"model_configs/{DEFAULT_CONFIG_NAME}")

# Optional shard parameters populated by Slurm array jobs.
selected_config_id = None
selected_config_index = None
selected_ticker_index = None
selected_seed_index = None
array_job_id = ""
array_task_id = 0

# Static DeepHit Architecture Grid Search (Split-Parquet Pipeline)

This architecture grid-search notebook uses static split parquet inputs (`train`/`val`/`test`) with fixed loss hyperparameters (ALPHA, BETA_L3) from loss tuning, and evaluates with unweighted static metrics.

In [ ]:
import gc
import importlib
import importlib.util
import json
import os
import random

import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import find_dotenv, load_dotenv
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

from src.models import (
    DeepHitMambaCompeting,
    DeepHitRNNCompeting,
    DeepHitRNNTransformerCompeting,
    DeepHitTransformerCompeting,
)
from src.notebook_data import (
    LabTransform,
    apply_dynamic_normalizer,
    build_order_batch_indices,
    extract_lob_features,
    extract_toxicity_features,
    recensor_after_horizon,
)
from src.notebook_evaluation import (
    standard_brier_score,
    uninformed_brier_score,
    uninformed_cif_curve_from_train,
)
from src.notebook_losses import static_deephit_total_loss

DATASET_BASE_DIR = Path("/ocean/projects/cis260122p/shared/data/datasets")
DATASET_STEM_TEMPLATE = "labeled_dataset_XNAS_ITCH_{ticker}_mbo_20251001_20260101"

REQUIRED_META_KEYS = ["t_max"]
REQUIRED_SPLIT_COLUMNS = {
    "entry_time",
    "duration_s",
    "event_type",
    "side",
    "entry_representation",
    "toxicity_representation",
}

EVENT_NAMES = ["FAVORABLE_FILL", "TOXIC_FILL"]
EVENT_CODES = [1, 2]

CONFIG_JSONL_PATH = str(
    Path(CONFIG_JSONL_PATH)
    if Path(CONFIG_JSONL_PATH).is_absolute()
    else (repo_root / CONFIG_JSONL_PATH)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.float16
LOADER_PIN_MEMORY = device.type != "cpu"
AUTOCAST_DEVICE = "cuda" if device.type == "cuda" else "cpu"

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Python:", sys.executable)

mamba_requested = False
if selected_config_id is not None or selected_config_index is not None:
    mamba_requested = True
elif model_name is not None and str(model_name).lower() == "mamba":
    mamba_requested = True

if mamba_requested:
    def _safe_import_status(module_name: str) -> tuple[bool, str | None, str | None]:
        try:
            module = importlib.import_module(module_name)
            return True, str(getattr(module, "__version__", "unknown")), None
        except Exception as exc:
            return False, None, f"{type(exc).__name__}: {exc}"

    causal_ok, causal_ver, causal_err = _safe_import_status("causal_conv1d")
    mamba_ok, mamba_ver, mamba_err = _safe_import_status("mamba_ssm")
    print("causal_conv1d:", causal_ver if causal_ok else f"broken ({causal_err})")
    print("mamba_ssm:", mamba_ver if mamba_ok else f"broken ({mamba_err})")

dotenv_path = find_dotenv(filename=".env", usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path, override=False)
else:
    load_dotenv(override=False)

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
if WANDB_ENABLE and WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb.login(key=WANDB_API_KEY, relogin=True)


def _ticker_paths(ticker_symbol: str) -> dict[str, Path]:
    symbol = str(ticker_symbol).upper()
    stem = DATASET_STEM_TEMPLATE.format(ticker=symbol)
    return {
        "train": DATASET_BASE_DIR / f"{stem}_train.parquet",
        "val": DATASET_BASE_DIR / f"{stem}_val.parquet",
        "test": DATASET_BASE_DIR / f"{stem}_test.parquet",
        "dynamic_manifest_meta": DATASET_BASE_DIR / f"{stem}_dynamic_manifest_meta.json",
    }


def _load_split_parquet(parquet_path: Path, split_name: str, ticker_symbol: str) -> pd.DataFrame:
    if not parquet_path.exists():
        raise FileNotFoundError(f"[{ticker_symbol}] Missing split parquet: {parquet_path}")

    read_columns = sorted(REQUIRED_SPLIT_COLUMNS | {"entry_representation_raw_top5"})
    try:
        df_split = pd.read_parquet(parquet_path, columns=read_columns)
    except Exception:
        df_split = pd.read_parquet(parquet_path, columns=sorted(REQUIRED_SPLIT_COLUMNS))

    if "entry_representation" not in df_split.columns:
        if "entry_representation_raw_top5" in df_split.columns:
            df_split = df_split.copy()
            df_split["entry_representation"] = df_split["entry_representation_raw_top5"]
        else:
            raise ValueError(
                f"[{ticker_symbol}] Missing entry_representation column in {split_name} split. "
                "Expected entry_representation or entry_representation_raw_top5."
            )

    missing_cols = sorted(REQUIRED_SPLIT_COLUMNS - set(df_split.columns))
    if missing_cols:
        raise ValueError(
            f"[{ticker_symbol}] Missing required columns in {split_name} split: {missing_cols}"
        )

    df_split = df_split.copy()
    df_split["dataset_split"] = split_name
    return df_split


def _event_codes_from_df(df: pd.DataFrame, ticker_symbol: str) -> np.ndarray:
    if "event_type_competing" in df.columns:
        events = df["event_type_competing"].to_numpy(dtype=np.int64, copy=False)
    elif "event_type" in df.columns:
        events = df["event_type"].to_numpy(dtype=np.int64, copy=False)
    else:
        raise ValueError(f"[{ticker_symbol}] Could not find event_type or event_type_competing column.")
    return events


def _fit_feature_normalizer(x_train: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    feat_mean = x_train.mean(axis=(0, 1), keepdims=True).astype(np.float32)
    feat_std = x_train.std(axis=(0, 1), keepdims=True).astype(np.float32)
    feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)

    side_idx = x_train.shape[2] - 2
    mask_idx = x_train.shape[2] - 1
    feat_mean[..., side_idx] = 0.0
    feat_std[..., side_idx] = 1.0
    feat_mean[..., mask_idx] = 0.0
    feat_std[..., mask_idx] = 1.0

    return feat_mean, feat_std


def _load_manifest_meta(meta_path: Path, ticker_symbol: str) -> tuple[dict, dict, float]:
    with open(meta_path, "r", encoding="utf-8") as f:
        manifest_meta = json.load(f)

    missing_keys = [k for k in REQUIRED_META_KEYS if k not in manifest_meta]
    if missing_keys:
        raise ValueError(f"[{ticker_symbol}] Missing keys in manifest metadata: {missing_keys}")

    t_max = float(manifest_meta["t_max"])
    if (not np.isfinite(t_max)) or t_max <= 0.0:
        raise ValueError(f"[{ticker_symbol}] Invalid t_max in metadata: {manifest_meta['t_max']!r}")

    preprocessing_config = manifest_meta.get("preprocessing_config")
    if not isinstance(preprocessing_config, dict):
        preprocessing_config = {}

    return manifest_meta, preprocessing_config, t_max


def _integrated_brier_score_to_tmax(bs_curve, time_grid) -> float:
    t_grid = np.asarray(time_grid, dtype=np.float64)
    bs_arr = np.asarray(bs_curve, dtype=np.float64)
    if t_grid.size == 0 or bs_arr.size == 0:
        return float("nan")

    t_min = float(t_grid[0])
    t_max_grid = float(t_grid[-1])
    span = t_max_grid - t_min
    if span <= 0.0:
        return float("nan")

    trapz_fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return float(trapz_fn(bs_arr, t_grid) / span)


def safe_pr_auc_binary(y_true_binary, y_score):
    if int(y_true_binary.sum()) == 0:
        return float("nan"), np.array([1.0]), np.array([0.0])
    precision, recall, _ = precision_recall_curve(y_true_binary, y_score)
    pr_auc = float(auc(recall[::-1], precision[::-1]))
    return pr_auc, precision, recall


def safe_roc_auc_binary(y_true_binary, y_score):
    if np.unique(y_true_binary).size < 2:
        return float("nan")
    return float(roc_auc_score(y_true_binary, y_score))


def _set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _load_config_records(config_jsonl_path: str) -> list[dict]:
    path = Path(config_jsonl_path)
    if not path.exists():
        raise FileNotFoundError(f"Config JSONL not found: {path}")

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if "config_id" not in record:
                raise ValueError(f"Missing config_id on line {line_no}")
            if "model_name" not in record:
                raise ValueError(f"Missing model_name on line {line_no}")
            records.append(record)

    if not records:
        raise ValueError(f"No config records found in {path}")
    return records


def _select_config_record(config_records: list[dict], config_by_id: dict[str, dict]) -> dict:
    if selected_config_id is not None and str(selected_config_id).strip():
        cfg_id = str(selected_config_id).strip()
        if cfg_id not in config_by_id:
            raise KeyError(f"selected_config_id={cfg_id} not found in config file")
        return config_by_id[cfg_id]

    if selected_config_index is not None:
        idx = int(selected_config_index)
        if idx < 0 or idx >= len(config_records):
            raise IndexError(f"selected_config_index={idx} out of range")
        return config_records[idx]

    raise ValueError("Either selected_config_id or selected_config_index must be provided for architecture tuning.")


def _build_model(
    config_record: dict,
    feature_dim: int,
    seq_len: int,
    output_steps: int,
) -> torch.nn.Module:
    model_name_local = str(config_record["model_name"]).lower()
    hidden_size = int(config_record["hidden_size"])
    fc_hidden = int(config_record.get("fc_hidden", int(round(hidden_size * config_record["fc_hidden_ratio"]))))
    fc_dropout = float(config_record.get("fc_dropout", 0.2))

    if model_name_local == "gru":
        return DeepHitRNNCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=hidden_size,
            num_layers=int(config_record["gru_layers"]),
            rnn_dropout=float(config_record.get("rnn_dropout", 0.2)),
            fc_hidden=fc_hidden,
            fc_dropout=fc_dropout,
        ).to(device)

    if model_name_local == "gru_transformer":
        return DeepHitRNNTransformerCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=hidden_size,
            num_layers=int(config_record["gru_layers"]),
            rnn_dropout=float(config_record.get("rnn_dropout", 0.2)),
            transformer_layers=int(config_record["transformer_layers"]),
            transformer_heads=int(config_record.get("transformer_heads", 4)),
            transformer_ff_dim=int(round(hidden_size * float(config_record.get("transformer_ff_ratio", 3.0)))),
            transformer_dropout=float(config_record.get("transformer_dropout", 0.1)),
            max_seq_len=seq_len,
            fc_hidden=fc_hidden,
            fc_dropout=fc_dropout,
        ).to(device)

    if model_name_local == "transformer":
        return DeepHitTransformerCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=hidden_size,
            num_layers=int(config_record["transformer_layers"]),
            num_heads=int(config_record.get("transformer_heads", 4)),
            transformer_ff_dim=int(round(hidden_size * float(config_record.get("transformer_ff_ratio", 3.0)))),
            transformer_dropout=float(config_record.get("transformer_dropout", 0.1)),
            max_seq_len=seq_len,
            fc_hidden=fc_hidden,
            fc_dropout=fc_dropout,
        ).to(device)

    if model_name_local == "mamba":
        if importlib.util.find_spec("mamba_ssm") is None:
            raise ImportError("mamba_ssm is not installed or not importable.")
        return DeepHitMambaCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=hidden_size,
            num_mamba_layers=int(config_record["mamba_layers"]),
            d_state=int(config_record.get("mamba_d_state", 16)),
            d_conv=int(config_record.get("mamba_d_conv", 4)),
            expand=int(config_record.get("mamba_expand", 2)),
            mamba_dropout=float(config_record.get("mamba_dropout", 0.15)),
            fc_hidden=fc_hidden,
            fc_dropout=fc_dropout,
        ).to(device)

    raise ValueError(f"Unsupported model_name: {model_name_local}")


TICKER_CONTEXT_CACHE: dict[str, dict] = {}


def _build_ticker_context(ticker_symbol: str) -> dict:
    symbol = str(ticker_symbol).upper()
    paths = _ticker_paths(symbol)

    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"[{symbol}] Missing files:\n" + "\n".join(missing))

    df_train = _load_split_parquet(paths["train"], "train", symbol)
    df_val = _load_split_parquet(paths["val"], "val", symbol)
    _ = _load_split_parquet(paths["test"], "test", symbol)

    manifest_meta, preprocessing_config, t_max = _load_manifest_meta(
        paths["dynamic_manifest_meta"],
        symbol,
    )

    lookback_steps = int(preprocessing_config.get("lookback_steps", 500))
    lob_feature_dim = int(preprocessing_config.get("lob_feature_dim", 20))
    tox_feature_dim = int(preprocessing_config.get("tox_feature_dim", 12))
    artifact_num_time_steps = int(preprocessing_config.get("num_time_steps", NUM_TIME_STEPS))

    if artifact_num_time_steps != int(NUM_TIME_STEPS):
        raise ValueError(
            f"[{symbol}] num_time_steps mismatch: notebook={NUM_TIME_STEPS}, "
            f"artifact={artifact_num_time_steps}"
        )

    x_train_lob = extract_lob_features(df_train, lookback_steps, feat_dim=lob_feature_dim)
    x_train_tox = extract_toxicity_features(df_train, lookback_steps, feat_dim=tox_feature_dim)
    x_train = np.concatenate([x_train_lob, x_train_tox], axis=2)

    x_val_lob = extract_lob_features(df_val, lookback_steps, feat_dim=lob_feature_dim)
    x_val_tox = extract_toxicity_features(df_val, lookback_steps, feat_dim=tox_feature_dim)
    x_val = np.concatenate([x_val_lob, x_val_tox], axis=2)

    y_train = df_train["duration_s"].to_numpy(dtype=np.float32, copy=False)
    d_train = _event_codes_from_df(df_train, symbol)
    y_val = df_val["duration_s"].to_numpy(dtype=np.float32, copy=False)
    d_val = _event_codes_from_df(df_val, symbol)

    y_train_h, d_train_h, n_train_recensored = recensor_after_horizon(y_train, d_train, t_max)
    y_val_h, d_val_h, n_val_recensored = recensor_after_horizon(y_val, d_val, t_max)

    feat_mean, feat_std = _fit_feature_normalizer(x_train)
    x_train_norm = apply_dynamic_normalizer(x_train, feat_mean, feat_std)
    x_val_norm = apply_dynamic_normalizer(x_val, feat_mean, feat_std)

    label_transform = LabTransform(NUM_TIME_STEPS, scheme="quantiles")
    y_train_disc, d_train_disc = label_transform.fit_transform(y_train_h.copy(), d_train_h.copy())
    y_val_disc, d_val_disc = label_transform.transform(y_val_h.copy(), d_val_h.copy())

    time_grid = np.asarray(label_transform.cuts, dtype=np.float32)
    if time_grid.size == 0:
        raise ValueError(f"[{symbol}] Empty time_grid produced by label_transform.")

    if x_train_norm.shape[0] == 0 or x_val_norm.shape[0] == 0:
        raise ValueError(f"[{symbol}] Empty train/val split after preprocessing.")

    order_keys_train = np.arange(x_train_norm.shape[0], dtype=np.int64)
    order_keys_val = np.arange(x_val_norm.shape[0], dtype=np.int64)

    context = {
        "ticker": symbol,
        "manifest_meta": manifest_meta,
        "paths": paths,
        "X_train": x_train_norm,
        "X_val": x_val_norm,
        "train_sample_idx": np.arange(x_train_norm.shape[0], dtype=np.int64),
        "val_sample_idx": np.arange(x_val_norm.shape[0], dtype=np.int64),
        "Y_train": y_train_h.astype(np.float32, copy=False),
        "D_train": d_train_h.astype(np.int64, copy=False),
        "Y_val": y_val_h.astype(np.float32, copy=False),
        "D_val": d_val_h.astype(np.int64, copy=False),
        "Y_train_t": torch.tensor(y_train_disc, dtype=torch.int64, device="cpu"),
        "D_train_t": torch.tensor(d_train_disc, dtype=torch.int64, device="cpu"),
        "ORDER_KEYS_TRAIN": order_keys_train,
        "ORDER_KEYS_TRAIN_t": torch.tensor(order_keys_train, dtype=torch.int64, device="cpu"),
        "Y_val_t": torch.tensor(y_val_disc, dtype=torch.int64, device="cpu"),
        "D_val_t": torch.tensor(d_val_disc, dtype=torch.int64, device="cpu"),
        "ORDER_KEYS_VAL": order_keys_val,
        "feat_mean": feat_mean,
        "feat_std": feat_std,
        "time_grid": time_grid,
        "output_steps": int(time_grid.shape[0]),
        "feature_dim": int(x_train_norm.shape[2]),
        "seq_len": int(x_train_norm.shape[1]),
        "aux_target_dim": int(x_train_norm.shape[2] - 2),
        "t_max": float(t_max),
        "n_train_recensored": int(n_train_recensored),
        "n_val_recensored": int(n_val_recensored),
    }

    print(
        f"[{symbol}] loaded train={x_train_norm.shape[0]:,}, val={x_val_norm.shape[0]:,}, "
        f"feature_dim={context['feature_dim']}, seq_len={context['seq_len']}, "
        f"output_steps={context['output_steps']}, t_max={context['t_max']:.6g}, "
        f"recensored(train/val)=({context['n_train_recensored']}/{context['n_val_recensored']})"
    )
    return context


def get_ticker_context(ticker_symbol: str) -> dict:
    symbol = str(ticker_symbol).upper()
    if symbol not in TICKER_CONTEXT_CACHE:
        TICKER_CONTEXT_CACHE[symbol] = _build_ticker_context(symbol)
    return TICKER_CONTEXT_CACHE[symbol]


def _iter_train_batches(context: dict, orders_per_batch: int, seed: int):
    batch_indices = build_order_batch_indices(
        context["ORDER_KEYS_TRAIN"],
        orders_per_batch=orders_per_batch,
        shuffle=True,
        seed=seed,
    )
    for idx_np in batch_indices:
        idx_t = torch.as_tensor(idx_np, dtype=torch.int64, device="cpu")
        x_b_cpu = torch.from_numpy(context["X_train"][idx_np])

        yield (
            x_b_cpu,
            context["Y_train_t"].index_select(0, idx_t),
            context["D_train_t"].index_select(0, idx_t),
            context["ORDER_KEYS_TRAIN_t"].index_select(0, idx_t),
        )


def _static_ctd_event(
    cif_event: np.ndarray,
    durations: np.ndarray,
    events: np.ndarray,
    event_code: int,
    time_grid_local: np.ndarray,
    eps: float = 1e-12,
) -> float:
    n = len(durations)
    if n == 0:
        return float("nan")

    tau_idx = np.searchsorted(time_grid_local, durations, side="left")
    tau_idx = np.clip(tau_idx, 0, len(time_grid_local) - 1)

    num = 0.0
    den = 0.0

    anchors = np.where(events == int(event_code))[0]
    for i in anchors:
        later_idx = np.where(durations > durations[i])[0]
        if later_idx.size == 0:
            continue

        tau_anchor = int(tau_idx[i])
        s_i = float(cif_event[tau_anchor, i])
        s_j = cif_event[tau_anchor, later_idx]

        concordant = (s_i > s_j).astype(np.float64)
        ties = (s_i == s_j).astype(np.float64)
        num += float(np.sum(concordant + 0.5 * ties))
        den += float(later_idx.size)

    if den <= eps:
        return float("nan")
    return num / den


def compute_validation_metrics(base_net: torch.nn.Module, context: dict) -> dict:
    base_net.eval()
    n_val = int(context["X_val"].shape[0])
    output_steps = int(context["output_steps"])
    cif_val = np.empty((len(EVENT_CODES), output_steps, n_val), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, n_val, BATCH_SIZE):
            x_b_np = context["X_val"][start : start + BATCH_SIZE]
            x_b = torch.from_numpy(x_b_np).to(device, non_blocking=LOADER_PIN_MEMORY)
            logits_b = base_net(x_b)
            pmf_b = torch.softmax(logits_b.reshape(logits_b.size(0), -1), dim=1).reshape(
                logits_b.size(0),
                len(EVENT_CODES),
                output_steps,
            )
            cif_b = torch.cumsum(pmf_b, dim=2).cpu().numpy()
            end = start + cif_b.shape[0]
            cif_val[:, :, start:end] = np.transpose(cif_b, (1, 2, 0))

    metrics = {}

    ctd_scores = {}
    for k, (event_code, event_name) in enumerate(zip(EVENT_CODES, EVENT_NAMES)):
        ctd_scores[event_name] = _static_ctd_event(
            cif_event=cif_val[k],
            durations=context["Y_val"],
            events=context["D_val"],
            event_code=int(event_code),
            time_grid_local=context["time_grid"],
        )

    ctd_values = np.asarray([ctd_scores.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ctd_macro = float(np.nanmean(ctd_values)) if not np.all(np.isnan(ctd_values)) else float("nan")

    metrics["val_ctd_macro"] = ctd_macro
    metrics["val_ctd_favorable"] = float(ctd_scores.get("FAVORABLE_FILL", np.nan))
    metrics["val_ctd_toxic"] = float(ctd_scores.get("TOXIC_FILL", np.nan))

    n_fav = int((context["D_val"] == 1).sum())
    n_tox = int((context["D_val"] == 2).sum())
    denom = n_fav + n_tox
    weighted_ctd = (
        (n_fav * metrics["val_ctd_favorable"] + n_tox * metrics["val_ctd_toxic"]) / denom
        if denom > 0 else float("nan")
    )
    metrics["val_weighted_ctd"] = float(weighted_ctd)

    p_fav_train_curve = uninformed_cif_curve_from_train(
        context["Y_train"],
        context["D_train"],
        1,
        context["time_grid"],
    )
    p_tox_train_curve = uninformed_cif_curve_from_train(
        context["Y_train"],
        context["D_train"],
        2,
        context["time_grid"],
    )

    bs_curves = {}
    for k, (event_code, event_name) in enumerate(zip(EVENT_CODES, EVENT_NAMES)):
        bs_curves[event_name] = standard_brier_score(
            durations=context["Y_val"],
            events=context["D_val"],
            cif_k=cif_val[k],
            event_code=event_code,
            time_grid=context["time_grid"],
        )

    bs_curves_uninformed = {
        "FAVORABLE_FILL": uninformed_brier_score(
            context["Y_val"],
            context["D_val"],
            p_fav_train_curve,
            1,
            context["time_grid"],
        ),
        "TOXIC_FILL": uninformed_brier_score(
            context["Y_val"],
            context["D_val"],
            p_tox_train_curve,
            2,
            context["time_grid"],
        ),
    }

    ibs_scores = {}
    ibs_scores_uninformed = {}
    for event_name, bs_arr in bs_curves.items():
        ibs_scores[event_name] = _integrated_brier_score_to_tmax(
            bs_arr,
            context["time_grid"],
        )
        ibs_scores_uninformed[event_name] = _integrated_brier_score_to_tmax(
            bs_curves_uninformed[event_name],
            context["time_grid"],
        )

    ibs_vals = np.asarray([ibs_scores.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ibs_vals_u = np.asarray([ibs_scores_uninformed.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ibs_macro = float(np.nanmean(ibs_vals)) if not np.all(np.isnan(ibs_vals)) else float("nan")
    ibs_macro_uninf = float(np.nanmean(ibs_vals_u)) if not np.all(np.isnan(ibs_vals_u)) else float("nan")

    metrics["val_ibs_macro"] = ibs_macro
    metrics["val_ibs_macro_uninformed"] = ibs_macro_uninf
    metrics["val_ibs_favorable"] = float(ibs_scores.get("FAVORABLE_FILL", np.nan))
    metrics["val_ibs_toxic"] = float(ibs_scores.get("TOXIC_FILL", np.nan))

    weighted_ibs = (
        (n_fav * metrics["val_ibs_favorable"] + n_tox * metrics["val_ibs_toxic"]) / denom
        if denom > 0 else float("nan")
    )
    weighted_ibs_uninformed = (
        (
            n_fav * float(ibs_scores_uninformed.get("FAVORABLE_FILL", np.nan))
            + n_tox * float(ibs_scores_uninformed.get("TOXIC_FILL", np.nan))
        ) / denom
        if denom > 0 else float("nan")
    )
    metrics["val_ibs_weighted"] = float(weighted_ibs)
    metrics["val_ibs_weighted_uninformed"] = float(weighted_ibs_uninformed)

    final_cif_tox = cif_val[1, -1, :]
    y_toxic = (context["D_val"] == 2).astype(int)
    ap_toxic, _, _ = safe_pr_auc_binary(y_toxic, final_cif_tox)
    auc_toxic = safe_roc_auc_binary(y_toxic, final_cif_tox)

    metrics["val_toxic_pr_auc"] = float(ap_toxic)
    metrics["val_toxic_roc_auc"] = float(auc_toxic)

    metrics["val_toxic_pr_auc_weighted"] = float(metrics["val_toxic_pr_auc"])
    metrics["val_toxic_pr_auc_sample"] = float(metrics["val_toxic_pr_auc"])
    metrics["val_toxic_roc_auc_weighted"] = float(metrics["val_toxic_roc_auc"])
    metrics["val_toxic_roc_auc_sample"] = float(metrics["val_toxic_roc_auc"])

    return metrics


def run_single_experiment(context: dict, config_record: dict, seed: int) -> dict:
    ticker = context["ticker"]
    _set_seed(seed)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    base_net = _build_model(
        config_record=config_record,
        feature_dim=int(context["feature_dim"]),
        seq_len=int(context["seq_len"]),
        output_steps=int(context["output_steps"]),
    )

    optimizer = torch.optim.Adam(base_net.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=LEARNING_RATE * 0.1,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    run_name = (
        f"{ticker}_seed{seed}_{str(config_record['model_name']).lower()}_{config_record['config_id']}"
    )

    wandb_run = None
    if WANDB_ENABLE:
        init_kwargs = {
            "project": WANDB_PROJECT,
            "name": run_name,
            "config": {
                "ticker": ticker,
                "seed": int(seed),
                "alpha": float(ALPHA),
                "beta_l3": float(BETA_L3),
                "sigma": float(SIGMA),
                **config_record,
                "learning_rate": float(LEARNING_RATE),
                "num_epochs": int(num_epochs),
                "orders_per_batch": int(ORDERS_PER_BATCH),
                "batch_size_eval": int(BATCH_SIZE),
                "t_max": float(context["t_max"]),
            },
            "mode": WANDB_MODE,
            "reinit": True,
        }
        if WANDB_ENTITY:
            init_kwargs["entity"] = WANDB_ENTITY
        wandb_run = wandb.init(**init_kwargs)

    for epoch in range(num_epochs):
        base_net.train()
        epoch_total = 0.0
        epoch_l1 = 0.0
        epoch_l2 = 0.0
        epoch_l3 = 0.0
        n_batches = 0

        for X_b_cpu, Y_b_cpu, D_b_cpu, O_b_cpu in _iter_train_batches(
            context,
            ORDERS_PER_BATCH,
            seed + epoch,
        ):
            X_b = X_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            Y_b = Y_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            D_b = D_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            O_b = O_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = base_net(X_b)
                total_loss, parts = static_deephit_total_loss(
                    logits=logits,
                    y=Y_b,
                    d=D_b,
                    order_ids=O_b,
                    x_batch=X_b,
                    base_net=base_net,
                    alpha=float(ALPHA),
                    sigma=float(SIGMA),
                    beta_l3=float(BETA_L3),
                    aux_target_dim=int(context["aux_target_dim"]),
                )

            if USE_AMP:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                optimizer.step()

            epoch_total += float(total_loss.item())
            epoch_l1 += float(parts["l1"].item())
            epoch_l2 += float(parts["l2"].item())
            epoch_l3 += float(parts["l3"].item())
            n_batches += 1

        if n_batches == 0:
            raise RuntimeError(f"[{ticker}] No training batches were produced.")

        scheduler.step()
        train_total = epoch_total / n_batches
        train_l1 = epoch_l1 / n_batches
        train_l2 = epoch_l2 / n_batches
        train_l3 = epoch_l3 / n_batches
        lr_now = float(optimizer.param_groups[0]["lr"])

        if wandb_run is not None:
            wandb_run.log(
                {
                    "epoch": epoch + 1,
                    "train/total_loss": train_total,
                    "train/l1_loss": train_l1,
                    "train/l2_loss": train_l2,
                    "train/l3_loss": train_l3,
                    "train/lr": lr_now,
                },
                step=epoch + 1,
            )

    metrics = compute_validation_metrics(base_net, context)

    result = {
        "ticker": ticker,
        "seed": int(seed),
        "alpha": float(ALPHA),
        "beta_l3": float(BETA_L3),
        **config_record,
        "num_epochs": int(num_epochs),
        "learning_rate": float(LEARNING_RATE),
        "run_name": run_name,
        **metrics,
    }

    if wandb_run is not None:
        wandb_eval = {f"eval/{k}": v for k, v in metrics.items()}
        wandb_run.log(wandb_eval)
        for k, v in metrics.items():
            wandb_run.summary[k] = v
        wandb_run.finish()

    return result

In [ ]:
TICKERS = [str(t).upper() for t in TICKERS]
if not TICKERS:
    raise ValueError("TICKERS must contain at least one symbol.")

CONFIG_RECORDS = _load_config_records(CONFIG_JSONL_PATH)
CONFIG_BY_ID = {record["config_id"]: record for record in CONFIG_RECORDS}
print(f"Loaded {len(CONFIG_RECORDS)} configs from {CONFIG_JSONL_PATH}")

if selected_config_id is not None or selected_config_index is not None:
    selected_config_records = [
        _select_config_record(CONFIG_RECORDS, CONFIG_BY_ID)
    ]

    ticker_idx = int(selected_ticker_index) if selected_ticker_index is not None else 0
    seed_idx = int(selected_seed_index) if selected_seed_index is not None else 0

    if ticker_idx < 0 or ticker_idx >= len(TICKERS):
        raise IndexError(f"selected_ticker_index={ticker_idx} out of range")
    if seed_idx < 0 or seed_idx >= len(SEEDS):
        raise IndexError(f"selected_seed_index={seed_idx} out of range")

    selected_tickers = [TICKERS[ticker_idx]]
    selected_seeds = [SEEDS[seed_idx]]
else:
    selected_config_records = CONFIG_RECORDS
    if model_name is not None and str(model_name).strip():
        selected_config_records = [
            r for r in selected_config_records
            if str(r["model_name"]).lower() == str(model_name).lower()
        ]
        if not selected_config_records:
            raise ValueError(f"No configs found for model_name={str(model_name).lower()}")

    selected_tickers = TICKERS
    selected_seeds = SEEDS

print("Tickers:", selected_tickers)
print("Seeds:", selected_seeds)
print("Alpha:", ALPHA)
print("Beta_l3:", BETA_L3)
print("Configs selected:", len(selected_config_records))

if selected_config_id is not None and str(selected_config_id).strip():
    print(
        f"Shard mode by config_id: {selected_config_id}, "
        f"ticker={selected_tickers[0]}, seed={selected_seeds[0]}"
    )
elif selected_config_index is not None:
    print(
        f"Shard mode by config_index: {selected_config_index}, "
        f"ticker={selected_tickers[0]}, seed={selected_seeds[0]}"
    )
else:
    print("Multi-config mode enabled.")

for ticker_symbol in selected_tickers:
    _ = get_ticker_context(ticker_symbol)

results = []
total_runs = len(selected_config_records) * len(selected_tickers) * len(selected_seeds)
run_idx = 0

for config_record in selected_config_records:
    config_id = str(config_record["config_id"])
    model_name_local = str(config_record["model_name"]).lower()
    print(f"\nConfig {config_id} ({model_name_local})")

    for ticker_symbol in selected_tickers:
        ticker_context = get_ticker_context(ticker_symbol)

        for seed in selected_seeds:
            run_idx += 1
            print(
                f"[{run_idx}/{total_runs}] "
                f"ticker={ticker_symbol}, seed={seed}, model={model_name_local}, config_id={config_id}"
            )

            run_result = run_single_experiment(
                context=ticker_context,
                config_record=config_record,
                seed=int(seed),
            )

            if str(array_job_id).strip():
                run_result["array_job_id"] = str(array_job_id).strip()
                run_result["array_task_id"] = int(array_task_id)

            results.append(run_result)

GRID_RESULTS_DF = pd.DataFrame(results)
if GRID_RESULTS_DF.empty:
    raise RuntimeError("No runs were completed.")

GRID_RESULTS_DF = GRID_RESULTS_DF.sort_values(
    "val_weighted_ctd", ascending=False
).reset_index(drop=True)

PER_TICKER_AGG_DF = (
    GRID_RESULTS_DF
    .groupby(["ticker", "config_id", "model_name"], as_index=False)
    .agg(
        mean_val_weighted_ctd=("val_weighted_ctd", "mean"),
        std_val_weighted_ctd=("val_weighted_ctd", "std"),
        mean_val_ibs_weighted=("val_ibs_weighted", "mean"),
        mean_val_toxic_pr_auc_weighted=("val_toxic_pr_auc_weighted", "mean"),
        runs=("val_weighted_ctd", "count"),
        trainable_params=("trainable_params", "first"),
        hidden_size=("hidden_size", "first"),
    )
)

BEST_PER_TICKER_DF = (
    PER_TICKER_AGG_DF
    .loc[PER_TICKER_AGG_DF.groupby("ticker")["mean_val_weighted_ctd"].idxmax()]
    .sort_values(["ticker", "mean_val_weighted_ctd"], ascending=[True, False])
    .reset_index(drop=True)
)

GLOBAL_AGG_DF = (
    GRID_RESULTS_DF
    .groupby(["config_id", "model_name"], as_index=False)
    .agg(
        mean_val_weighted_ctd=("val_weighted_ctd", "mean"),
        std_val_weighted_ctd=("val_weighted_ctd", "std"),
        mean_val_ibs_weighted=("val_ibs_weighted", "mean"),
        mean_val_toxic_pr_auc_weighted=("val_toxic_pr_auc_weighted", "mean"),
        runs=("val_weighted_ctd", "count"),
        trainable_params=("trainable_params", "first"),
        hidden_size=("hidden_size", "first"),
    )
    .sort_values("mean_val_weighted_ctd", ascending=False)
    .reset_index(drop=True)
)

BEST_GLOBAL_DF = GLOBAL_AGG_DF.head(20).copy()

print("\nTop individual runs by validation weighted C-td")
display(GRID_RESULTS_DF.head(20))

print("\nBest architecture per ticker (aggregated across seeds)")
display(BEST_PER_TICKER_DF)

print("\nTop global architectures (aggregated across all tickers and seeds)")
display(BEST_GLOBAL_DF)

def _safe_tag(value: str) -> str:
    return str(value).replace("/", "_").replace(" ", "_")

array_job_id_str = str(array_job_id).strip()
array_task_id_str = str(array_task_id).strip() if array_task_id is not None else ""
shard_label = f"{array_job_id_str}_{array_task_id_str}" if array_job_id_str else "local"

if len(selected_config_records) == 1:
    config_tag = _safe_tag(selected_config_records[0]["config_id"])
else:
    config_tag = "multi"

results_root = repo_root / "results"
shard_dir = results_root / "arch_grid_search_shards"
shard_dir.mkdir(parents=True, exist_ok=True)

shard_stem = f"static_arch_grid_results_{config_tag}_{shard_label}"
shard_csv_path = shard_dir / f"{shard_stem}.csv"
shard_meta_path = shard_dir / f"{shard_stem}.json"

GRID_RESULTS_DF.to_csv(shard_csv_path, index=False)

shard_meta = {
    "array_job_id": array_job_id_str,
    "array_task_id": array_task_id_str,
    "config_jsonl_path": str(CONFIG_JSONL_PATH),
    "selected_config_id": str(selected_config_records[0]["config_id"]) if len(selected_config_records) == 1 else "",
    "selected_config_index": int(selected_config_index) if selected_config_index is not None else None,
    "alpha": float(ALPHA),
    "beta_l3": float(BETA_L3),
    "tickers": [str(t) for t in selected_tickers],
    "seeds": [int(s) for s in selected_seeds],
    "rows": int(len(GRID_RESULTS_DF)),
    "csv_path": str(shard_csv_path),
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
with open(shard_meta_path, "w", encoding="utf-8") as f:
    json.dump(shard_meta, f, indent=2)

print(f"\nSaved shard results: {shard_csv_path}")
print(f"Saved shard metadata: {shard_meta_path}")